# 08 Interpretability

What a trained model does inside, mainly for `subhalo_C_mixed` (one model for spacings 3–10 h⁻¹ Mpc), with a single-spacing model for contrast. Inference only.

1. **Activation maps**: the same patch of the box at λ = 3 and λ = 10, through an early layer, the bottleneck and the last layer.
2. **Layer-by-layer similarity (CKA)**: where in the network the representations of different spacings become alike.
3. **Can λ be read out?** A linear probe for the tracer spacing at each layer.
4. **Effective receptive field**: how far from a voxel the model actually looks, per spacing.
5. **Errors, uncertainty and calibration** on the test slab: maps, accuracy against distance to the nearest tracer, and reliability.

Runtime: **L4 GPU** with **High-RAM**. About 15–20 minutes.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
token = userdata.get('GITHUB_TOKEN')
!rm -rf /content/deepvoid
!git clone -q "https://{token}@github.com/skumagai3/deepvoid.git" /content/deepvoid
!git -C /content/deepvoid remote set-url origin https://github.com/skumagai3/deepvoid.git
!pip -q install --force-reinstall --no-deps /content/deepvoid
del token
!git -C /content/deepvoid log --oneline -1
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from deepvoid import Catalog
from deepvoid import interpret as I
from deepvoid.calibrate import reliability
from deepvoid.config import load_config
from deepvoid.inputs import build
from deepvoid.labels import CLASS_NAMES
from deepvoid.models import load_model
from deepvoid.predict import predict_volume
from deepvoid.report import Report
from deepvoid.split import Region

root = Path('/content/drive/MyDrive/deepvoid_runs')
RUN = 'subhalo_C_mixed'                 # the model to look inside
COMPARE = 'subhalo_A_scratch_L10'       # a single-spacing model for contrast; None to skip
SPACINGS = [3, 5, 7, 10]
NAMES = {L: f'tng300-3/subhalos/L{L}' for L in SPACINGS}
COLORS = {3: '#1f77b4', 5: '#ff7f0e', 7: '#2ca02c', 10: '#d62728'}
CELL = 205.0 / 512                      # h^-1 Mpc per cell
rng = np.random.default_rng(0)
catalog = Catalog()


def load_run(name):
    out = root / name
    cfg = load_config(out / 'config.yaml')
    regions = {k: Region(**v) for k, v in json.loads((out / 'split.json').read_text()).items()}
    norms = json.loads((out / 'normalization.json').read_text())
    return cfg, regions, norms, load_model(out / 'best.keras')


def inputs_for(cfg, regions, norms):
    """Each spacing's input, normalized the way this run sees it."""
    train = regions['train'].mask(512)
    return {L: build(catalog, NAMES[L], cfg['data']['smoothing'], norms, train) for L in SPACINGS}


cfg, regions, norms, model = load_run(RUN)
X = inputs_for(cfg, regions, norms)
labels = catalog.load(cfg['data']['labels']).data
test = regions['test']
layers = I.block_names(model)
print('blocks:', layers)
rep = Report(root / RUN / 'interpretability')   # figures and numbers saved here
print('saving to', rep.folder)

## 1. Activation maps

One 128³ patch, middle slice. For each block, the three channels that vary most at λ = 10, shown for the same patch at λ = 3 (top) and λ = 10 (bottom). If the network turns both spacings into the same internal picture, the deep rows should look alike even when the inputs do not.

In [ ]:
origin = [(200, 200, 200)]
patch = {L: I.periodic_crops(X[L], origin, 128) for L in (3, 10)}
show = [layers[0], 'bottom_relu2', layers[-1]]
acts = {L: I.activations(model, patch[L], show) for L in (3, 10)}

fig, axes = plt.subplots(2, 1 + 3 * len(show), figsize=(3 * (1 + 3 * len(show)), 6.5))
for r, L in enumerate((3, 10)):
    ax = axes[r, 0]
    ax.imshow(patch[L][0, :, :, 64, 0].T, origin='lower', cmap='magma')
    ax.set_title(f'input, λ = {L}')
    for b, name in enumerate(show):
        a = acts[L][name][0]
        k = a.shape[2] // 2
        top = np.argsort(acts[10][name][0].reshape(-1, a.shape[-1]).std(0))[::-1][:3]
        for j, ch in enumerate(top):
            ax = axes[r, 1 + 3 * b + j]
            ax.imshow(a[:, :, k, ch].T, origin='lower', cmap='viridis')
            ax.set_title(f'{name.split("_")[0]} ch{ch}, λ = {L}', fontsize=9)
for ax in axes.ravel():
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); rep.show(fig, '1_activation_maps')

## 2. Where do spacings become alike? (CKA)

Linear CKA between activations for the same voxels at two spacings: 1 means the same representation up to rotation and scale. Each curve compares λ = 3, 5 or 7 against λ = 10, block by block. Solid lines are this model; dashed, the single-spacing model for contrast. A curve that starts low and rises means the network maps different inputs onto a shared description of the web.

In [ ]:
def cka_curves(model, X, names, n_crops=8, size=64, per_layer=4000):
    size = min(size, test.size // 8 * 8)      # crops stay inside the test slab
    origins = [(test.start + rng.integers(0, test.size - size + 1),
                rng.integers(0, 512), rng.integers(0, 512)) for _ in range(n_crops)]
    crops = {L: I.periodic_crops(X[L], origins, size) for L in SPACINGS}
    acts = {L: {'input': crops[L], **I.activations(model, crops[L], names)} for L in SPACINGS}
    curves = {}
    for L in SPACINGS[:-1]:
        vals = []
        for name in ['input'] + names:
            a, b = acts[L][name], acts[10][name]
            flat_a, flat_b = a.reshape(-1, a.shape[-1]), b.reshape(-1, b.shape[-1])
            idx = np.random.default_rng(1).choice(len(flat_a), min(per_layer, len(flat_a)), replace=False)
            vals.append(I.linear_cka(flat_a[idx], flat_b[idx]))
        curves[L] = vals
    return curves

curves = cka_curves(model, X, layers)
compare = None
if COMPARE:
    c_cfg, c_regions, c_norms, c_model = load_run(COMPARE)
    c_X = inputs_for(c_cfg, c_regions, c_norms)
    compare = cka_curves(c_model, c_X, I.block_names(c_model))

ticks = ['input'] + [n.split('_')[0] for n in layers]
fig, ax = plt.subplots(figsize=(9, 4.5))
for L, vals in curves.items():
    ax.plot(ticks, vals, '-o', color=COLORS[L], label=f'λ = {L} vs 10 ({RUN})')
    if compare:
        ax.plot(ticks, compare[L], '--', color=COLORS[L], alpha=0.6, label=f'λ = {L} vs 10 ({COMPARE})')
ax.set_ylabel('linear CKA'); ax.set_ylim(0, 1.02); ax.legend(fontsize=8); ax.grid(alpha=0.3)
rep.show(fig, '2_cka')
rep.record(cka={'blocks': ticks, RUN: curves, **({COMPARE: compare} if compare else {})})

## 3. Can the spacing be read out of each layer?

A ridge regression from each block's pooled features (channel means and standard deviations per crop) to log λ, cross-validated R². High at the input is trivial (sparser fields have more empty cells). What matters is whether the deep layers keep that information, i.e. whether the network tracks how sparse its input is, or discards it.

In [ ]:
crops, target = [], []
for L in SPACINGS:
    origins = [tuple(rng.integers(0, 512, 3)) for _ in range(24)]
    crops.append(I.periodic_crops(X[L], origins, 64))
    target += [np.log(L)] * 24
crops, target = np.concatenate(crops), np.array(target)
feats = I.activations(model, crops, layers, reduce=I.pooled)
r2 = [I.probe_r2(I.pooled(crops), target)] + [I.probe_r2(feats[n], target) for n in layers]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(ticks, r2, '-o', color='k')
ax.set_ylabel('cross-validated R² for log λ'); ax.set_ylim(min(0, min(r2)) - 0.05, 1.02); ax.grid(alpha=0.3)
rep.show(fig, '3_spacing_probe')
rep.record(spacing_probe_r2=dict(zip(ticks, r2)))
print({t: round(v, 3) for t, v in zip(ticks, r2)})

## 4. Effective receptive field

How much the predicted class at the centre of a 128³ patch depends on the input at each distance (mean absolute gradient in shells, averaged over a few patches). Left: the profile, normalized at the centre. Right: the radius that holds 50% and 90% of the total dependence. If the model reaches further for sparser tracers, the radii grow with λ.

In [ ]:
def erf(model, X, L, n=4):
    origins = [tuple(rng.integers(0, 512, 3)) for _ in range(n)]
    g = I.input_gradient(model, I.periodic_crops(X[L], origins, 128), spacing=float(L))
    idx = np.indices(g.shape) - 64
    r = np.sqrt((idx ** 2).sum(0)).astype(int).ravel()
    mass = np.bincount(r, weights=g.ravel())
    cum = np.cumsum(mass) / mass.sum()
    rr, prof = I.radial_profile(g)
    return rr, prof / prof[0], np.searchsorted(cum, 0.5), np.searchsorted(cum, 0.9)

fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
radii = {}
for L in SPACINGS:
    rr, prof, r50, r90 = erf(model, X, L)
    radii[L] = (r50 * CELL, r90 * CELL)
    ax[0].semilogy(rr * CELL, prof, color=COLORS[L], label=f'λ = {L}')
ax[0].set_xlabel('distance from voxel [h⁻¹ Mpc]'); ax[0].set_ylabel('|gradient|, normalized'); ax[0].legend()
ax[1].plot(SPACINGS, [radii[L][0] for L in SPACINGS], '-o', label='50%')
ax[1].plot(SPACINGS, [radii[L][1] for L in SPACINGS], '-o', label='90%')
ax[1].set_xlabel('λ [h⁻¹ Mpc]'); ax[1].set_ylabel('radius [h⁻¹ Mpc]'); ax[1].legend()
rep.show(fig, '4_receptive_field')
rep.record(receptive_field_radii_mpc={L: {'r50': a, 'r90': b} for L, (a, b) in radii.items()})
print({L: (round(a, 1), round(b, 1)) for L, (a, b) in radii.items()})

## 5. Errors, uncertainty and calibration on the test slab

Predicts the whole box once per spacing (about a minute each) and keeps only what the plots need.

- **Maps**: a slice through the middle of the test slab: truth, prediction, wrong voxels, and the entropy of the predicted probabilities (0 = certain, 2 bits = uniform over the four classes).
- **Accuracy against distance to the nearest tracer**: where the model fails as the tracers thin out.
- **Reliability**: whether a voxel predicted with 80% confidence is right 80% of the time; ECE is the average gap.

In [ ]:
k = test.start + test.size // 2            # yz-slice through the middle of the test slab
edges = np.arange(0, 21, 1.0)              # distance bins, h^-1 Mpc
truth_test = labels[test.start:test.stop]
maps, binned, rel = {}, {}, {}
for L in SPACINGS:
    probs = predict_volume(lambda b: model.predict_on_batch(I.model_input(model, b, L)), X[L], crop=128, margin=16, batch_size=4)
    p_test = probs[test.start:test.stop]
    del probs
    pred_test = p_test.argmax(-1)
    s = k - test.start
    maps[L] = dict(pred=pred_test[s], entropy=I.entropy(p_test[s]), raw=X[L][k, ..., 0])
    dist = I.distance_to_tracer(catalog.load(NAMES[L]).data, test) * CELL
    binned[L] = I.binned_scores(dist, pred_test, truth_test, edges)
    sub = rng.choice(p_test[..., 0].size, 2_000_000, replace=False)
    rel[L] = reliability(p_test.reshape(-1, 4)[sub], truth_test.ravel()[sub])
    print(f'λ = {L}: test accuracy {(pred_test == truth_test).mean():.3f}, ECE {rel[L]["ece"]:.3f}')
    del p_test, pred_test, dist
rep.record(test_slab={L: {'ece': rel[L]['ece'], 'accuracy_by_distance': binned[L]['accuracy'],
                          'recall_by_distance': {CLASS_NAMES[c]: binned[L][f'recall_{c}'] for c in range(4)},
                          'voxels_by_distance': binned[L]['counts']} for L in SPACINGS},
           distance_bin_edges_mpc=edges)


In [ ]:
from matplotlib.colors import ListedColormap
cmap_mask = ListedColormap(['#2b6cb0', '#9ae6b4', '#f6ad55', '#c53030'])
truth = labels[k]
fig, axes = plt.subplots(len(SPACINGS), 4, figsize=(16, 4 * len(SPACINGS)))
for r, L in enumerate(SPACINGS):
    m = maps[L]
    axes[r, 0].imshow(m['raw'].T, origin='lower', cmap='magma'); axes[r, 0].set_title(f'input, λ = {L}')
    axes[r, 1].imshow(m['pred'].T, origin='lower', cmap=cmap_mask, vmin=-0.5, vmax=3.5, interpolation='nearest')
    axes[r, 1].set_title('prediction')
    axes[r, 2].imshow(np.where(m['pred'] != truth, truth, np.nan).T, origin='lower', cmap=cmap_mask,
                      vmin=-0.5, vmax=3.5, interpolation='nearest')
    axes[r, 2].set_facecolor('#eeeeee'); axes[r, 2].set_title('wrong voxels (true class)')
    im = axes[r, 3].imshow(m['entropy'].T, origin='lower', cmap='Greys', vmin=0, vmax=2)
    axes[r, 3].set_title('entropy [bits]')
for ax in axes.ravel():
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); rep.show(fig, '5_test_slab_maps')

fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(truth.T, origin='lower', cmap=cmap_mask, vmin=-0.5, vmax=3.5, interpolation='nearest')
ax.set_title('truth (same slice)'); ax.set_xticks([]); ax.set_yticks([])
rep.show(fig, '5_test_slab_truth')

In [ ]:
centers = 0.5 * (edges[1:] + edges[:-1])
fig, ax = plt.subplots(1, 3, figsize=(20, 4.5))
for L in SPACINGS:
    ax[0].plot(centers, binned[L]['accuracy'], '-o', ms=3, color=COLORS[L], label=f'λ = {L}')
    ax[1].plot(centers, binned[L]['counts'] / binned[L]['counts'].sum(), '-', color=COLORS[L])
ax[0].set_xlabel('distance to nearest tracer [h⁻¹ Mpc]'); ax[0].set_ylabel('accuracy'); ax[0].legend()
ax[1].set_xlabel('distance to nearest tracer [h⁻¹ Mpc]'); ax[1].set_ylabel('fraction of test voxels')
for c, name in enumerate(CLASS_NAMES):
    ax[2].plot(centers, binned[10][f'recall_{c}'], '-o', ms=3, label=name, color=cmap_mask(c))
ax[2].set_xlabel('distance to nearest tracer [h⁻¹ Mpc]'); ax[2].set_ylabel('recall at λ = 10'); ax[2].legend()
rep.show(fig, '5_accuracy_vs_distance')

fig, ax = plt.subplots(figsize=(5.5, 5))
for L in SPACINGS:
    mids = 0.5 * (rel[L]['bin_edges'][1:] + rel[L]['bin_edges'][:-1])
    ax.plot(mids, rel[L]['accuracy'], '-o', ms=3, color=COLORS[L], label=f'λ = {L}, ECE {rel[L]["ece"]:.3f}')
ax.plot([0, 1], [0, 1], 'k:', lw=1)
ax.set_xlabel('predicted confidence'); ax.set_ylabel('fraction correct'); ax.legend()
rep.show(fig, '5_reliability')

Everything above is saved in `deepvoid_runs/<RUN>/interpretability/` (one PNG per figure and `results.json`). Tell Claude when it has finished.